<a href="https://colab.research.google.com/github/Asav23/Text-Summariser/blob/main/Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1️⃣ Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# 2️⃣ Set the path to your folder
project_path = '/content/drive/MyDrive/Summariser/'

# 3️⃣ Change current working directory to your project folder
import os
os.chdir(project_path)
print("Current working directory:", os.getcwd())


Current working directory: /content/drive/MyDrive/Summariser


In [ ]:
# 1️⃣ Make sure dependencies are loaded


# 2️⃣ Import your classes from existing notebooks
%run TextEmbedder.ipynb
%run IdeaExtractor.ipynb
%run SemanticClusterer.ipynb
%run ClusterRanker.ipynb
%run BARTRealizer.ipynb
%run SemanticSummarizer.ipynb

# Fix: Monkey-patch BARTRealizer to call BART directly, bypassing the broken pipeline.
from transformers import BartTokenizer, BartForConditionalGeneration

def fixed_bart_realizer_init(self, model_name="facebook/bart-large-cnn"):
    self.tokenizer = BartTokenizer.from_pretrained(model_name)
    self.model = BartForConditionalGeneration.from_pretrained(model_name)

def fixed_bart_realizer_realize(self, ideas, max_length=60):
    if not ideas:
        return ""
    prompt = ", ".join(ideas)
    inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    summary_ids = self.model.generate(inputs["input_ids"], max_length=max_length, min_length=20, do_sample=False)
    return self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)

BARTRealizer.__init__ = fixed_bart_realizer_init
BARTRealizer.realize = fixed_bart_realizer_realize

# 3️⃣ Prompt user for paragraph
input_text = input("Paste your paragraph here and press Enter:\n")

# 4️⃣ Initialize pipeline and summarize
summarizer = SemanticSummarizer(use_bart=True)
summary = summarizer.summarize(input_text)

# 5️⃣ Print the final summary
print("=== Final Summary ===")
print(summary)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 4.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


Paste your paragraph here and press Enter:
The quiet rhythm of everyday life often hides moments of unexpected clarity, where even the smallest experiences can shift our perspective in meaningful ways. A brief conversation, a change in routine, or a sudden challenge can reveal strengths we didn’t realize we had, pushing us to grow beyond our comfort zones. Over time, these subtle shifts accumulate, shaping who we are and how we see the world. While it’s easy to overlook these moments in the rush of daily responsibilities, taking the time to reflect on them can offer a deeper sense of purpose and direction.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

=== Final Summary ===
Life hides moments unexpected, everyday life hides moments, responsibilities taking time reflect, our comfort zones, the quiet rhythm, quiet rhythm everyday life.


In [ ]:
# 6️⃣ Similarity Test - compare summary to original text
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Embed both texts using your existing TextEmbedder
orig_embedding = embedder.embed_document(input_text)
summary_embedding = embedder.embed_document(summary)

# Cosine similarity
similarity = cosine_similarity(
    orig_embedding.reshape(1, -1),
    summary_embedding.reshape(1, -1)
)[0][0]

# Metadata report
print("\n=== Similarity Metadata ===")
print(f"Original length  : {len(input_text.split())} words")
print(f"Summary length   : {len(summary.split())} words")
print(f"Compression ratio: {len(summary.split()) / len(input_text.split()):.2%}")
print(f"Cosine similarity: {similarity:.4f}")

if similarity >= 0.85:
    print("Quality          : ✅ Excellent — summary is semantically close to original")
elif similarity >= 0.70:
    print("Quality          : ⚠️  Good — some meaning may be lost")
else:
    print("Quality          : ❌ Poor — summary diverges significantly from original")

NameError: name 'embedder' is not defined